<a href="https://colab.research.google.com/github/brianellis1997/Sarcasm_Detection/blob/main/GloVe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sarcasm Detection using NLP Techniques
### Team 2: Brian Ellis, Lindsey Rich, Elena Kern

## Load Libraries and Dataset

In [ ]:
import pandas as pd
import torch
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

#test comment

In [ ]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.optim as optim
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

In [ ]:
!git clone https://github.com/brianellis1997/Sarcasm_Detection.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
train_bal = pd.read_csv('/content/drive/MyDrive/Sarcasm_Data/Train_Balanced.csv')   # Make sure path is correct in your google drive
train_bal.head()

In [ ]:
train_bal.info()

In [ ]:
train_bal['label'].value_counts()

We can see our dataset is balanced.

## GLOVE

### 6B Tokens, 400k tokens, 100 Dimensions

In [ ]:
import numpy as np
from nltk.tokenize import word_tokenize
from gensim.models import KeyedVectors
import nltk
nltk.download('punkt')

# Load GloVe word vectors
glove_file = '/content/drive/MyDrive/Sarcasm_Data/glove.6B.100d.txt'  # Adjust the path to your downloaded GloVe file
word_vectors = KeyedVectors.load_word2vec_format(glove_file, binary=False, no_header=True)

# Function to obtain word vectors for a sentence
def get_sentence_embedding(sentence):
    words = word_tokenize(sentence)  # Tokenize sentence into words
    word_embeddings = []
    for word in words:
        try:
            word_embeddings.append(word_vectors[word])  # Get GloVe vector for word
        except KeyError:
            # Handle out-of-vocabulary words
            pass
    if len(word_embeddings) == 0:
        # Handle case where all words are out-of-vocabulary
        return np.zeros(word_vectors.vector_size)
    else:
        # Aggregate word vectors (take sum)
        return np.sum(word_embeddings, axis=0)

# Apply to comment and parent comment
comment_vectors = train_bal['comment'].apply(get_sentence_embedding)
parent_comment_vectors = train_bal['parent_comment'].apply(get_sentence_embedding)


In [ ]:
train_bal['comment_vector'] = comment_vectors
train_bal['parent_vector'] = parent_comment_vectors
train_bal.info()

In [ ]:
# Flatten the comment vectors into separate variables
comment_vector_cols = [f'comment_vector_{i+1}' for i in range(100)]
for i, col in enumerate(comment_vector_cols):
    train_bal[col] = train_bal['comment_vector'].apply(lambda x: x[i])

# Flatten the parent comment vectors into separate variables
parent_comment_vector_cols = [f'parent_comment_vector_{i+1}' for i in range(100)]
for i, col in enumerate(parent_comment_vector_cols):
    train_bal[col] = train_bal['parent_vector'].apply(lambda x: x[i])

In [ ]:
train_bal.head()

Embed each word instead of comment.

In [ ]:
# Open the GloVe file and read the first few lines
with open(glove_file, 'r') as f:
    for i in range(5):
        print(f.readline())

In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import numpy as np

# Flatten the arrays
comment_vectors_array = np.array([vector for vector in comment_vectors])
parent_comment_vectors_array = np.array([vector for vector in parent_comment_vectors])

# Reduce dimensions using t-SNE
tsne = TSNE(n_components=2)
comment_tsne = tsne.fit_transform(comment_vectors_array)
parent_comment_tsne = tsne.fit_transform(parent_comment_vectors_array)

# Visualize embeddings using scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(comment_tsne[:, 0], comment_tsne[:, 1], label='Comments')
plt.scatter(parent_comment_tsne[:, 0], parent_comment_tsne[:, 1], label='Parent Comments')
plt.title('t-SNE Visualization')
plt.legend()

# Save the plot as an image file
plt.savefig('tsne_plot.png')

plt.show()

In [ ]:
import pickle

# Save comment and parent vectors using Pickle
with open('comment_vectors.pkl', 'wb') as f:
    pickle.dump(comment_vectors, f)

with open('parent_comment_vectors.pkl', 'wb') as f:
    pickle.dump(parent_comment_vectors, f)

In [ ]:
# Load comment and parent vectors using Pickle
with open('comment_vectors.pkl', 'rb') as f:
    comment_vectors = pickle.load(f)

with open('parent_comment_vectors.pkl', 'rb') as f:
    parent_comment_vectors = pickle.load(f)

In [ ]:
comment_vectors

In [ ]:
comment_vectors[0]

Take row sums to get 100 dimensional vector back.

For random forest and svm, PCA or use lower dimensional embeddings.
<br>Try stochastic gradient descent.
<br>Mini batching with random forest.
<br>Try naive bayes?
<br>Feed sentence embeddings directly into model so 100dim comment and 100dim parent comment instead of higher dimensional tfidf vectors.

In [ ]:
comment_vectors[0].shape

In [ ]:
comment_vectors[0].sum()

100 dimensional arrays

In [ ]:
# Calculate the sum of vectors for each comment in comment_vectors
comment_sums = [np.sum(vector) for vector in comment_vectors]

comment_sums[0]

In [ ]:
train_bal['comment_sums'] = comment_sums

In [ ]:
train_bal.info()

In [ ]:
# Calculate the sum of vectors for each comment in comment_vectors
parent_sums = [np.sum(vector) for vector in parent_comment_vectors]

parent_sums[0]

In [ ]:
train_bal['parent_sums'] = parent_sums

In [ ]:
train_bal.info()

In [ ]:
train_bal.head()

In [ ]:
train_bal.to_csv('Train_Balanced_Sums.csv')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
X = train_bal.drop(columns=['comment', 'author', 'label', 'subreddit', 'score', 'ups', 'downs', 'date', 'created_utc', 'Unnamed: 0', 'parent_comment', 'comment_vector', 'parent_vector'])
y = train_bal['label']

In [ ]:
X.head()

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=22)

print(X_train.shape)
print(X_val.shape)

In [ ]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_val = sc.transform(X_val)

In [ ]:
logit = LogisticRegression(C=1, n_jobs=-1, solver='lbfgs',
                           random_state=22, verbose=1)

logit.fit(X_train, y_train)
valid_pred = logit.predict(X_val)

from sklearn.metrics import accuracy_score
accuracy_score(y_val, valid_pred)

In [ ]:
# Maybe try tfidf vectorizer function with comment and then the 100dim embeddings with parent comment

### 840B tokens, 2.2M vocab, 300 dimensional vectors

In [ ]:
import numpy as np
from nltk.tokenize import word_tokenize
from gensim.models import KeyedVectors
import nltk
nltk.download('punkt')

# Load GloVe word vectors
glove_file = '/content/drive/MyDrive/Sarcasm_Data/glove.840B.300d.txt'  # Adjust the path to your downloaded GloVe file
word_vectors = KeyedVectors.load_word2vec_format(glove_file, binary=False, no_header=True)

# Function to obtain word vectors for a sentence
def get_sentence_embedding(sentence):
    words = word_tokenize(sentence)  # Tokenize sentence into words
    word_embeddings = []
    for word in words:
        try:
            word_embeddings.append(word_vectors[word])  # Get GloVe vector for word
        except KeyError:
            # Handle out-of-vocabulary words
            pass
    if len(word_embeddings) == 0:
        # Handle case where all words are out-of-vocabulary
        return np.zeros(word_vectors.vector_size)
    else:
        # Aggregate word vectors (take sum)
        return np.sum(word_embeddings, axis=0)

# Apply to comment and parent comment
comment_vectors = train_bal['comment'].apply(get_sentence_embedding)
parent_comment_vectors = train_bal['parent_comment'].apply(get_sentence_embedding)


In [ ]:
train_bal['comment_vector'] = comment_vectors
train_bal['parent_vector'] = parent_comment_vectors
train_bal.info()

In [ ]:
# Flatten the comment vectors into separate variables
comment_vector_cols = [f'comment_vector_{i+1}' for i in range(100)]
for i, col in enumerate(comment_vector_cols):
    train_bal[col] = train_bal['comment_vector'].apply(lambda x: x[i])

# Flatten the parent comment vectors into separate variables
parent_comment_vector_cols = [f'parent_comment_vector_{i+1}' for i in range(100)]
for i, col in enumerate(parent_comment_vector_cols):
    train_bal[col] = train_bal['parent_vector'].apply(lambda x: x[i])

In [ ]:
train_bal.head()

Embed each word instead of comment.

In [ ]:
# # Open the GloVe file and read the first few lines
# with open(glove_file, 'r') as f:
#     for i in range(5):
#         print(f.readline())

In [ ]:
# import matplotlib.pyplot as plt
# from sklearn.manifold import TSNE
# import numpy as np

# # Flatten the arrays
# comment_vectors_array = np.array([vector for vector in comment_vectors])
# parent_comment_vectors_array = np.array([vector for vector in parent_comment_vectors])

# # Reduce dimensions using t-SNE
# tsne = TSNE(n_components=2)
# comment_tsne = tsne.fit_transform(comment_vectors_array)
# parent_comment_tsne = tsne.fit_transform(parent_comment_vectors_array)

# # Visualize embeddings using scatter plot
# plt.figure(figsize=(8, 6))
# plt.scatter(comment_tsne[:, 0], comment_tsne[:, 1], label='Comments')
# plt.scatter(parent_comment_tsne[:, 0], parent_comment_tsne[:, 1], label='Parent Comments')
# plt.title('t-SNE Visualization')
# plt.legend()

# # Save the plot as an image file
# plt.savefig('tsne_plot.png')

# plt.show()

In [ ]:
# import pickle

# # Save comment and parent vectors using Pickle
# with open('comment_vectors.pkl', 'wb') as f:
#     pickle.dump(comment_vectors, f)

# with open('parent_comment_vectors.pkl', 'wb') as f:
#     pickle.dump(parent_comment_vectors, f)

In [ ]:
# # Load comment and parent vectors using Pickle
# with open('comment_vectors.pkl', 'rb') as f:
#     comment_vectors = pickle.load(f)

# with open('parent_comment_vectors.pkl', 'rb') as f:
#     parent_comment_vectors = pickle.load(f)

In [ ]:
# comment_vectors

In [ ]:
# comment_vectors[0]

Take row sums to get 100 dimensional vector back.

For random forest and svm, PCA or use lower dimensional embeddings.
<br>Try stochastic gradient descent.
<br>Mini batching with random forest.
<br>Try naive bayes?
<br>Feed sentence embeddings directly into model so 100dim comment and 100dim parent comment instead of higher dimensional tfidf vectors.

In [ ]:
# comment_vectors[0].shape

In [ ]:
# comment_vectors[0].sum()

100 dimensional arrays

In [ ]:
# # Calculate the sum of vectors for each comment in comment_vectors
# comment_sums = [np.sum(vector) for vector in comment_vectors]

# comment_sums[0]

In [ ]:
# train_bal['comment_sums'] = comment_sums

In [ ]:
# train_bal.info()

In [ ]:
# # Calculate the sum of vectors for each comment in comment_vectors
# parent_sums = [np.sum(vector) for vector in parent_comment_vectors]

# parent_sums[0]

In [ ]:
# train_bal['parent_sums'] = parent_sums

In [ ]:
# train_bal.info()

In [ ]:
# train_bal.head()

In [ ]:
# train_bal.to_csv('Train_Balanced_Sums.csv')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
X = train_bal.drop(columns=['comment', 'author', 'label', 'subreddit', 'score', 'ups', 'downs', 'date', 'created_utc', 'Unnamed: 0', 'parent_comment', 'comment_vector', 'parent_vector'])
y = train_bal['label']

In [ ]:
X.head()

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_val.shape)

In [ ]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_val = sc.transform(X_val)

In [ ]:
logit = LogisticRegression(C=1, n_jobs=-1, solver='lbfgs',
                           random_state=22, verbose=1)

logit.fit(X_train, y_train)
valid_pred = logit.predict(X_val)

from sklearn.metrics import accuracy_score
accuracy_score(y_val, valid_pred)

In [ ]:
train_bal[:20]['comment']

In [ ]:
# Remove null rows
train_bal = train_bal[-train_bal['comment'].isnull()]
train_bal.info()

Now we're going to split our data into train and test sets using an 80/20 split

In [ ]:
df_shuffled = train_bal.sample(frac=1, random_state=22)  # Shuffle data

# Split into train and test (80% train, 20% test)
train_bal, test_bal = train_test_split(df_shuffled, test_size=0.2, random_state=22)

print(train_bal.shape)
print(test_bal.shape)

In [ ]:
train_bal['label'].value_counts(normalize=True)

In [ ]:
test_bal['label'].value_counts(normalize=True)

## EDA

In [ ]:
# Separating sarcastic vs non sarcastic comments
sarcastic_com = train_bal[train_bal['label'] == 1]
non_sarcastic_com = train_bal[train_bal['label'] == 0]

In [ ]:
# # WordCloud for sarcastic comments
# wordcloud = WordCloud(width=800, height=400, background_color='white').generate(' '.join(sarcastic_com['comment'].values.ravel()))

# # Display the generated word cloud using matplotlib
# plt.figure(figsize=(10, 5))
# plt.imshow(wordcloud, interpolation='bilinear')
# plt.axis('off')
# plt.show()

In [ ]:
# # WordCloud for non-sarcastic comments
# wordcloud = WordCloud(width=800, height=400, background_color='white').generate(' '.join(non_sarcastic_com['comment'].values.ravel()))

# # Display the generated word cloud using matplotlib
# plt.figure(figsize=(10, 5))
# plt.imshow(wordcloud, interpolation='bilinear')
# plt.axis('off')
# plt.show()

# Visualizing Sarcastic Comments by Subreddit

In [ ]:
subreddit_count = sarcastic_com['subreddit'].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
top = subreddit_count.head(20)
plt.figure(figsize=(10, 6))
sns.barplot(x=top.values, y=top.index)
plt.xlabel('Number of Sarcastic Comments')
plt.ylabel('Subreddit')
plt.title('Frequency of Sarcastic Comments by Subreddit')
plt.show()


In [ ]:
import nltk
from nltk import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem.porter import *
nltk.download('punkt')
nltk.download('stopwords')

import string

import gensim
from gensim.models.phrases import Phraser, Phrases
from gensim.models.word2vec import Word2Vec

import spacy # for a lemmatization example

from sklearn.manifold import TSNE

import pandas as pd
from bokeh.io import output_notebook, output_file
from bokeh.plotting import show, figure

In [ ]:
comments = train_bal['comment']

In [ ]:
comments.info()

In [ ]:
comments.dropna(inplace=True)
comments.info()

In [ ]:
!pip install dask[delayed]

In [ ]:
import dask.dataframe as dd
from dask import delayed

In [ ]:
comment_test = comments.head(10000)

#Create large function that does this text preprocessing so that it can be used with X_train, also figure out how to split and keep track of data


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
def preprocess(text):
  tokenized_comments = batch_tokenize(text.tolist())
  phrases = Phrases(tokenized_comments)
  bigram = Phraser(phrases)
  bigram_comments = [bigram[comment] for comment in tokenized_comments]
  tokenizer = Tokenizer(lower=False)
  tokenizer.fit_on_texts(bigram_comments)
  word_index = tokenizer.word_index
  word_size = len(word_index) + 1

  # uses tokenizer object with the created word index to make each comment a list of integers corresponding to their index
  sequences = tokenizer.texts_to_sequences(bigram_comments)


  return word_index, word_size, sequences



In [ ]:
#  Replace this?
def batch_tokenize(texts, batch_size=50000):
    tokenized_texts = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        tokenized_batch = [word_tokenize(text) for text in batch]
        tokenized_texts.extend(tokenized_batch)
    return tokenized_texts

#tokenized_comments = batch_tokenize(comments.tolist())

In [ ]:
# len(tokenized_comments)

In [ ]:
#phrases = Phrases(tokenized_comments)

In [ ]:
# bigram = Phraser(phrases)

In [ ]:
# bigram.phrasegrams

In [ ]:
# bigram_comments = [bigram[comment] for comment in tokenized_comments]

In [ ]:
# from tensorflow.keras.preprocessing.text import Tokenizer

# tokenizer = Tokenizer(lower=False)
# tokenizer.fit_on_texts(bigram_comments)
# word_index = tokenizer.word_index

# word_index is a dictionary where keys are words and values are their corresponding indices

In [ ]:
# len(word_index)

In [ ]:
# word_size = len(word_index) + 1

In [ ]:
# Adjust parameters as needed
size = 150
window = 7
min_count = 15
workers = 4
sg = 1
negative = 5
epochs = 7


model = Word2Vec(bigram_comments, vector_size=size, window=window, min_count=min_count, workers=workers)

# model.save("/content/drive/Sarcasm_Data/model/word2vec.w2v")


In [ ]:
# model.save("model_embedding.w2v")

In [ ]:
import gensim

# skip re-training the model with the next line:
model = gensim.models.Word2Vec.load('/content/drive/MyDrive/word2vec.model')

In [ ]:
model.wv['dog']

In [ ]:
similar_words = model.wv.most_similar('ok')
similar_words

In [ ]:
model.wv.doesnt_match("mother father sister brother dog".split())

In [ ]:
model.wv.similarity('father', 'dog')

In [ ]:
model.wv.most_similar(positive=['father', 'woman'], negative=['man'])   # Should expect mother

In [ ]:
model.wv.most_similar(positive=['husband', 'woman'], negative=['man'])  # Should expect wife

### t-SNE Word Embedding Visualization

In [ ]:
# tsne = TSNE(n_components=2, n_iter=1000)

In [ ]:
# X_2d = tsne.fit_transform(model.wv[model.wv.key_to_index])
# coords_df = pd.DataFrame(X_2d, columns=['x','y'])
# coords_df['token'] = model.wv.index_to_key
# coords_df.head()


In [ ]:
# coords_df.to_csv("/content/drive/MyDrive/Sarcasm_Data/coords_df", index=False)

In [ ]:
coords_df = pd.read_csv('/content/drive/MyDrive/Sarcasm_Data/coords_df')
output_notebook()
subset_df = coords_df.sample(n=5000)
p = figure(min_width=800, min_height=800)
_ = p.text(x=subset_df.x, y=subset_df.y, text=subset_df.token)
show(p)

# Start LSTM architecture with these initial embeddings before BERT


# Need to create embeddings matrix for LSTM


In [ ]:
train_bal.dropna(inplace=True)

In [ ]:
train_bal.shape[0]

In [ ]:
train_bal.tail(5)

In [ ]:
from sklearn.model_selection import train_test_split
# split the dataset into 60% training and 40% validation and test sets
train_df, val_test_df = train_test_split(train_bal, test_size=0.4, random_state=42)

# split the 40% into half validation and half test sets
val_df, test_df = train_test_split(val_test_df, test_size=0.5, random_state=42)

In [ ]:
X_train = train_df['comment']
y_train = train_df['label']

In [ ]:
word2vec = Word2Vec.load("/content/drive/MyDrive/word2vec.model")


In [ ]:
#extracting pretrained weights for embedding layer using word2vec model
word_vectors = word2vec.wv


In [ ]:
vector_size = model.vector_size
vector_size

In [ ]:
word_index, word_size, _ = preprocess(comments)

In [ ]:
#inititlze embeddings matrix
import numpy as np
embedding_matrix = np.zeros((word_size, vector_size))

In [ ]:
#fill embeddings matrix for LSTM with tensor flow : each row should correspond to a word
for word, i in word_index.items():
    if word in word_vectors:
        embedding_matrix[i] = word_vectors[word]


In [ ]:
embedding_df = pd.DataFrame(embedding_matrix)

In [ ]:
embedding_df.head(5)

In [ ]:
train_index,_, X_train = preprocess(X_train)

In [ ]:
X_train

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train_padded = pad_sequences(X_train, maxlen=500, padding='post')


In [ ]:
df = pd.DataFrame(X_train_padded)
df.head(5)

In [ ]:
# import tensorflow as tf
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import LSTM, Dense, Embedding
# from tensorflow.keras.preprocessing.sequence import pad_sequences

# model = Sequential()
# model.add(Embedding(input_dim=word_size, output_dim=word2vec.vector_size, weights=[embedding_matrix], trainable=False))
# model.add(LSTM(128))
# model.add(Dense(1, activation='sigmoid'))


# model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


# model.fit(X_train_padded, y_train, epochs=10, validation_split=0.2)

# # get X_train data into format where the embeddings layer can look up its corresposnding vector, add padding so that all input is the same length
# # model.evaluate(X_test, y_test)


# New Try

In [ ]:
import tensorflow
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Embedding, SpatialDropout1D, LSTM
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.callbacks import ModelCheckpoint
import os
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

In [ ]:
word_size = embedding_matrix.shape[0]
embedding_size = embedding_matrix.shape[1]
max_comment_length = X_train_padded.shape[1]

In [ ]:
model = Sequential()


model.add(Embedding(input_dim=word_size,
                    output_dim=embedding_size,
                    weights=[embedding_matrix],
                    input_length=max_comment_length,
                    trainable=False))

model.add(Bidirectional(LSTM(128)))

model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# default learning rate .001
# add attention / dropout regularization


model.fit(X_train_padded, y_train, epochs=6, batch_size=128, validation_split=0.2)





In [ ]:
model.save('/content/drive/MyDrive/model.h5')

In [ ]:
from tensorflow.keras.models import load_model
import os
first_model = load_model('/content/drive/MyDrive/model.h5')